In [ ]:
import pandas as pd
import joblib
from tqdm import tqdm
tqdm.pandas()

In [ ]:
df = pd.read_csv(f"data/LIAR/train.tsv", sep='\t', header=None)
df['content'] = df[2]
df['type'] = df[1]
df['id'] = df[0].map(lambda x: 0) # necessary for cleaning pipeline
df['title'] = df[5] # necessary for cleaning pipeline
df.head(3)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,content,type,id,title
0,2635.json,false,Says the Annies List political group supports ...,abortion,dwayne-bohac,State representative,Texas,republican,0.0,1.0,0.0,0.0,0.0,a mailer,Says the Annies List political group supports ...,false,0,State representative
1,10540.json,half-true,When did the decline of coal start? It started...,"energy,history,job-accomplishments",scott-surovell,State delegate,Virginia,democrat,0.0,0.0,1.0,1.0,0.0,a floor speech.,When did the decline of coal start? It started...,half-true,0,State delegate
2,324.json,mostly-true,"Hillary Clinton agrees with John McCain ""by vo...",foreign-policy,barack-obama,President,Illinois,democrat,70.0,71.0,160.0,163.0,9.0,Denver,"Hillary Clinton agrees with John McCain ""by vo...",mostly-true,0,President


In [6]:
from sklearn.pipeline import Pipeline
from sklearn_transformers import FilterTransformer
from sklearn_transformers import CleaningTransformer

clean_pipeline = Pipeline([
    ("filter", FilterTransformer(
        drop_cols=range(14),
        remove_nulls_col_names=["type", "content"],
        deduplicate_cols=["content"],
        convert_to_category_cols=["type"],
        remove_cols_with_value={})),
    ("clean", CleaningTransformer())
])

In [7]:
cleaned = clean_pipeline.fit_transform(df)

In [8]:
reliables = ['true', 'mostly-true']
cleaned.type.unique()

<ArrowExtensionArray>
['false', 'half-true', 'mostly-true', 'true', 'barely-true', 'pants-fire']
Length: 6, dtype: dictionary<values=large_string, indices=int64, ordered=0>[pyarrow]

In [13]:
tokenizer = joblib.load("data/preprocessed/tokenizers/top10k.joblib")

pd.DataFrame({
    'content' : cleaned['content'],
    'is_reliable' : cleaned['type'].map(lambda t: t in reliables)
}).to_csv(f"data/LIAR/cleaned_train.csv")